## Extracting the complexity measures 

In [1]:
import os, random, tqdm, torch, warnings

In [2]:
import numpy as np
import pandas as pd

In [3]:
from itertools import islice
from multiprocessing import Pool
from pymfe.mfe import MFE
from tabicl_prior import sample_datasets

In [4]:
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [5]:
SEED  = 42

In [6]:
random.seed(SEED)
np.random.seed(SEED)

## Helper Functions

In [7]:
def normalize(file):
    df = pd.read_csv(file, na_values=['?', 'Not_Say'], low_memory=False)
    X, y = df.iloc[:, :-1], df.iloc[:, -1]

    X = X.dropna(axis=1, how='all')
    cat = X.select_dtypes(exclude='number').columns
    num = X.columns.difference(cat)

    X[num] = X[num].fillna(X[num].mean())
    X[cat] = X[cat].fillna('missing')

    return X.to_numpy(), y.to_numpy()

In [8]:
def load_datasets(path):
    return [(file, *normalize(f'{path}/{file}')) for file in os.listdir(path)]

In [9]:
def extract(args):
    i, X, y = args
    mfe = MFE(groups='complexity', features='all', summary='mean', random_state=SEED)
    mfe.fit(X, y, suppress_warnings=True)
    measures, values = mfe.extract(suppress_warnings=True)
    return {'id': i, 'n_classes': len(np.unique(y)), 'n_rows': X.shape[0], 'n_cols': X.shape[1], **dict(zip(measures, map(float, values)))}

## Real datasets

In [10]:
# data_tuples = load_datasets('tabarena/') + load_datasets('talent/')

In [ ]:
# with Pool(16) as pool:   
#     complexity = list(tqdm.tqdm(pool.imap(extract, data_tuples), total=len(data_tuples)))

In [12]:
# df = pd.DataFrame(complexity)
# df.to_csv('real.csv')

## Synthetic datasets

In [10]:
def run_stage(stage, n_datasets, jobs, prior, block):
    synthetic = sample_datasets('clf', stage=stage, n_datasets=n_datasets, seed=SEED, n_jobs=prior)
    data_tuples = ((i, d['X'], d['y']) for i, d in enumerate(synthetic))

    with Pool(jobs) as pool:
        while chunk := list(islice(data_tuples, block)):
            res = pool.map(extract, chunk)
            pd.DataFrame(res).to_csv(f'synthetic_stage{stage}.csv', mode='a', index=False,
                                     header=not os.path.exists(f'synthetic_stage{stage}.csv'))

In [13]:
run_stage(1, 32_000_000, 96, 32, 9600)
run_stage(2, 2_560_000, 120, 8, 6400)
run_stage(3, 640_000, 8, 4, 128)